# 05B. Ground-Station Decision Support & Acoustic Cross-Check Engine
### Sonobuoy Sound Detective: Offline Underwater Acoustic Event Detection
**Role in Project:** Operational Deployment Prototype (Hybrid ML + Deterministic Signal Rules)

This notebook simulates the offline sonobuoy ground-station operator workstation:
1. Ingesting raw acoustic audio streams.
2. Slicing into 4.0s windows & running event candidate detection.
3. Feature extraction & model inference using the winning classifier.
4. **Deterministic Acoustic Cross-Check Rules:**
   - Confidence threshold check (< 0.60 $ightarrow$ *Review Required*).
   - SNR estimate check (< 3.0 dB $ightarrow$ *Low Quality Warning*).
   - Tonal consistency validation: Flags vessel predictions lacking narrowband persistent tonals.
   - Ambient contradiction validation: Flags ambient predictions containing strong machinery tones.
5. Interactive Operator Decision Cards and feedback audit logging.


In [ ]:
import os
import sys
import json
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import joblib

from src.utils.config import (
    PROCESSED_DIR, RAW_DIR, SAMPLE_RATE, SEGMENT_DURATION,
    OVERLAP_FRACTION, BANDPASS_LOW_HZ, BANDPASS_HIGH_HZ,
    LOW_CONFIDENCE_THRESHOLD, MIN_SNR_DB
)
from src.preprocessing.audio_utils import to_mono, remove_dc_offset, apply_bandpass_filter, rms_normalize
from src.preprocessing.segmenter import segment_audio
from src.features.extractor import extract_all_features
from src.features.event_detection import detect_candidate_event

print("Decision support system loaded.")


## 1. Load Winning Model from Final Comparison


In [ ]:
model_search_paths = [
    Path("../04_person4_adaboost_and_xgboost/models/model_xgboost.joblib"),
    Path("../03_person3_decision_tree_and_random_forest/models/model_random_forest.joblib"),
    Path("../02_person2_svm_and_naive_bayes/models/model_svm_rbf.joblib"),
    Path("../01_person1_lr_and_knn/models/model_logistic_regression.joblib"),
]

model_path = None
for p in model_search_paths:
    if p.exists():
        model_path = p
        break

if not model_path:
    # Any available model
    any_models = list(Path("..").rglob("model*.joblib"))
    if any_models:
        model_path = any_models[0]

if not model_path:
    raise FileNotFoundError("No trained model found! Please execute an algorithm notebook first.")

print(f"Active Production Model: {model_path}")
model = joblib.load(model_path)

# Scaler
scaler_path = model_path.parent / model_path.name.replace("model_", "scaler_")
scaler = joblib.load(scaler_path) if scaler_path.exists() else None

with open(Path("../data/features/selected_features.json")) as f:
    raw_sel = json.load(f)
    selected_features = raw_sel["selected_features"] if isinstance(raw_sel, dict) and "selected_features" in raw_sel else raw_sel

classes = ["ambient", "unknown", "vessel_like"]
print(f"Model Type: {type(model).__name__} | Features: {len(selected_features)}")


## 2. Ingest Sample Mission Log Audio Recording


In [ ]:
audio_files = list(PROCESSED_DIR.rglob("*.wav"))
if not audio_files:
    audio_files = list(RAW_DIR.rglob("*.wav"))

mission_audio_path = audio_files[0]
print(f"Processing Hydrophone Stream: {mission_audio_path.name}")

y_raw, orig_sr = sf.read(str(mission_audio_path))
y_mono = to_mono(y_raw)
if orig_sr != SAMPLE_RATE:
    y_mono = librosa.resample(y_mono, orig_sr=orig_sr, target_sr=SAMPLE_RATE)

y_clean = remove_dc_offset(y_mono)
y_clean = apply_bandpass_filter(y_clean, sr=SAMPLE_RATE, low_hz=BANDPASS_LOW_HZ, high_hz=BANDPASS_HIGH_HZ)
y_clean = rms_normalize(y_clean)

segments = segment_audio(y_clean, sr=SAMPLE_RATE, window_sec=SEGMENT_DURATION, overlap=OVERLAP_FRACTION)
print(f"Audio Duration: {len(y_clean)/SAMPLE_RATE:.2f}s | Sliced into {len(segments)} analysis windows.")


## 3. Run Inference Pipeline with Deterministic Acoustic Cross-Check Rules


In [ ]:
def evaluate_acoustic_rules(pred_class, conf, evt, snr_est):
    warnings = []
    status = "Consistent"
    
    if conf < LOW_CONFIDENCE_THRESHOLD:
        warnings.append(f"Model confidence ({conf:.1%}) below required threshold ({LOW_CONFIDENCE_THRESHOLD:.0%}).")
        status = "Review Required"
        
    if snr_est < MIN_SNR_DB:
        warnings.append(f"Estimated SNR ({snr_est:.1f} dB) below minimum threshold ({MIN_SNR_DB} dB).")
        status = "Low Quality / Review Required"
        
    if pred_class == "vessel_like" and evt.get("tonal_persistence", 0) < 0.15:
        warnings.append("Vessel predicted, but weak/no persistent narrowband tonal peaks detected.")
        status = "Review Required (Acoustic Anomaly)"
        
    if pred_class == "ambient" and evt.get("tonal_persistence", 0) > 0.60:
        warnings.append("Ambient predicted, but strong persistent machinery tone detected.")
        status = "Review Required (Possible Missed Target)"
        
    return status, warnings

results = []
for i, seg in enumerate(segments):
    audio_seg = seg["audio"]
    evt = detect_candidate_event(audio_seg, sr=SAMPLE_RATE)
    feats = extract_all_features(audio_seg, sr=SAMPLE_RATE, clip_id=f"seg_{i}")
    snr_est = feats.get("snr_estimate", 10.0)
    
    x_vec = np.array([feats.get(col, 0.0) for col in selected_features]).reshape(1, -1)
    x_vec = np.nan_to_num(x_vec, nan=0.0)
    if scaler:
        x_vec = scaler.transform(x_vec)
        
    probs = model.predict_proba(x_vec)[0]
    pred_idx = np.argmax(probs)
    pred_class = classes[pred_idx] if pred_idx < len(classes) else "unknown"
    conf = probs[pred_idx]
    
    status, warns = evaluate_acoustic_rules(pred_class, conf, evt, snr_est)
    
    results.append({
        "Detection ID": f"DET-{i+1:03d}",
        "Window": f"{seg['start_sec']:.1f}s - {seg['end_sec']:.1f}s",
        "Prediction": pred_class,
        "Confidence": f"{conf:.1%}",
        "Dominant Freq": f"{evt.get('dominant_freq_hz', 0):.0f} Hz",
        "SNR (dB)": f"{snr_est:.1f}",
        "Rule Status": status,
        "Warnings": "; ".join(warns) if warns else "None",
        "_probs": dict(zip(classes, probs))
    })

df_detections = pd.DataFrame(results)
display(df_detections[["Detection ID", "Window", "Prediction", "Confidence", "Dominant Freq", "SNR (dB)", "Rule Status", "Warnings"]])


## 4. Operator Telemetry Card & Feedback Logging


In [ ]:
def show_telemetry_card(det):
    print("="*65)
    print(f"  OPERATOR DECISION TELEMETRY — {det['Detection ID']} ({det['Window']})")
    print("="*65)
    print(f"  Class Classification  : {det['Prediction'].upper()}")
    print(f"  Confidence            : {det['Confidence']}")
    print(f"  Dominant Frequency    : {det['Dominant Freq']}")
    print(f"  Estimated SNR         : {det['SNR (dB)']} dB")
    print(f"  Rule Engine Status    : {det['Rule Status']}")
    print(f"  Acoustic Warnings     : {det['Warnings']}")
    print("\n  Posterior Probabilities:")
    for cls, p in det['_probs'].items():
        bar = "█" * int(p * 20)
        print(f"    {cls:<14} [{p:>5.1%}] {bar}")
    print("="*65)

if results:
    show_telemetry_card(results[0])
    
    audit_entry = {
        "detection_id": results[0]["Detection ID"],
        "window": results[0]["Window"],
        "predicted_label": results[0]["Prediction"],
        "operator_decision": "ACCEPT",
        "corrected_label": results[0]["Prediction"],
        "notes": "Verified narrow-band shaft line at 180 Hz on LOFAR.",
        "operator": "acoustic_analyst_01"
    }
    print("\nAudit Log Record:")
    print(json.dumps(audit_entry, indent=2))
